In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
# transforms part
train_trainforms = transforms.Compose([
  transforms.Grayscale(num_output_channels = 1),
  transforms.Resize((48, 48)),
  transforms.RandomHorizontalFlip(p=0.4),
  transforms.RandomRotation(10),
  transforms.ColorJitter(brightness=0.3, contrast=0.3),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.5], std=[0.5])
])

val_transforms = transforms.Compose([
  transforms.Grayscale(num_output_channels=1),
  transforms.Resize((48, 48)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.5], std=[0.5])
])

In [3]:
train_dataset = datasets.ImageFolder('archive/train', transform=train_trainforms)
val_dataset = datasets.ImageFolder('archive/test', transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(train_dataset.class_to_idx)
print({cls: len([x for x in train_dataset.targets if x == idx])
       for cls, idx in train_dataset.class_to_idx.items()})

{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
{'angry': 3995, 'disgust': 436, 'fear': 4097, 'happy': 7215, 'neutral': 4965, 'sad': 4830, 'surprise': 3171}


In [4]:
class EmotionCNN(nn.Module):
  def __init__(self, num_classes = 7):
    super().__init__()

    self.features = nn.Sequential(
      # Block 1 -------(1, 48, 48) --> (32, 24, 24) -----------------
      nn.Conv2d(1, 32, kernel_size=3, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(inplace=True),
      nn.Conv2d(32, 32, kernel_size=3, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(inplace=True),
      nn.MaxPool2d(2, 2),
      nn.Dropout2d(0.25),


      # Block 2 ------(32, 24, 24) --> (64, 12, 12) ------------
      nn.Conv2d(32, 64, kernel_size=3, padding=1),
      nn.BatchNorm2d(64),
      nn.ReLU(inplace=True),
      nn.Conv2d(64, 64, kernel_size=3, padding=1),
      nn.BatchNorm2d(64),
      nn.ReLU(inplace=True),
      nn.MaxPool2d(2, 2),
      nn.Dropout2d(0.25),

      # Block 3 ------(64, 12, 12) --> (128, 6, 6) ---------------
      nn.Conv2d(64, 128, kernel_size=3, padding=1),
      nn.BatchNorm2d(128),
      nn.ReLU(inplace=True),
      nn.Conv2d(128, 128, kernel_size=3, padding=1),
      nn.BatchNorm2d(128),
      nn.ReLU(inplace=True),
      nn.MaxPool2d(2, 2),
      nn.Dropout2d(0.25)
    )

    self.classifier = nn.Sequential(
      nn.Flatten(),
      nn.Linear(128*6*6, 512),
      nn.ReLU(inplace=True),
      nn.Dropout(0.5),
      nn.Linear(512, 256),
      nn.ReLU(inplace=True),
      nn.Dropout(0.5),
      nn.Linear(256, num_classes)
    )
  
  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [5]:
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_counts = np.array([len([x for x in train_dataset.targets if x ==i]) for i in range(7)])

class_weights = 1/class_counts
class_weights = class_weights / class_weights.sum() * 7
class_weights = torch.FloatTensor(class_weights).to(device)

model = EmotionCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
  model.train()
  total_loss, correct, total = 0,0,0

  for img, labels in loader:
    img, labels = img.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(img)
    loss = criterion(outputs, labels)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    total_loss += loss.item()
    _, predicted = outputs.max(1)
    correct += predicted.eq(labels).sum().item()
    total += labels.size(0)

  return total_loss/len(loader), 100. * correct/total

def val_epoch(model, loader, criterion, device):
  model.eval()
  total_loss, correct, total = 0,0,0
  
  with torch.no_grad():
    for img, labels in loader:
      img, labels = img.to(device), labels.to(device)
      outputs = model(img)
      loss = criterion(outputs, labels)

      total_loss += loss.item()
      _, predicted = outputs.max(1)
      correct += predicted.eq(labels).sum().item()
      total += labels.size(0)

  return total_loss/len(loader), 100. * correct/total



In [7]:
# best_val_acc = 0
# num_epochs = 60

# for epoch in range(num_epochs):
#   train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
#   val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
#   scheduler.step()

#   print(f"Epoch {epoch} |"
#         f"train_loss {train_loss:.3f} acc {train_acc:.2f}% | \nval loss {val_loss:.3f} acc{val_loss:.2f}%")
  
#   if val_acc > best_val_acc:
#     best_val_acc = val_acc
#     torch.save(model.state_dict(), "best_emotion_model.pth")
#     print(f" --> saved new best: {val_acc:.2f}%")

In [8]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import mediapipe as mp
import urllib.request
import os

model_path = 'blaze_face_short_range.tflite'
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite',
        model_path
    )

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceDetectorOptions(
    base_options=base_options,
    min_detection_confidence=0.6
)
face_detector = vision.FaceDetector.create_from_options(options)

I0000 00:00:1775153841.741737  151005 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775153841.788845  151018 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.288.01), renderer: NVIDIA GeForce GTX 1650/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775153841.796461  151007 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
import cv2
import torch
import mediapipe as mp
import numpy as np
from PIL import Image
import urllib.request
import os
from collections import deque

model = EmotionCNN().to(device)
model.load_state_dict(torch.load('best_emotion_model.pth', map_location=device))
model.eval()

EMOTIONS = ['Angry', 'Disgusted', 'Fearful', 'Happy', 'Neutral', 'Sad', 'Surprised']

inference_transform = val_transforms

model_path = 'blaze_face_short_range.tflite'
if not os.path.exists(model_path):
    print("Downloading MediaPipe face detector model...")
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite',
        model_path
    )
    print("Done.")

# initialize face detector
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceDetectorOptions(
  base_options=base_options,
  min_detection_confidence=0.6
)
face_detector = vision.FaceDetector.create_from_options(options)

cap = cv2.VideoCapture('/dev/video1')

prediction_buffer = deque(maxlen=10)

if not cap.isOpened():
  print("Cannot open camera")
else:
  try:
    while True:
      ret, frame = cap.read()
      if not ret:
        print("Failed to grab frame")
        break

      h, w = frame.shape[:2]
      rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
      detection_result = face_detector.detect(mp_image)

      if detection_result.detections:
        for detection in detection_result.detections:
          bbox = detection.bounding_box
          x1 = max(0, bbox.origin_x)
          y1 = max(0, bbox.origin_y)
          x2 = min(w, bbox.origin_x + bbox.width)
          y2 = min(h, bbox.origin_y + bbox.height)

          face_crop = frame[y1:y2, x1:x2]
          if face_crop.size == 0: continue

          pil_face = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
          tensor = inference_transform(pil_face).unsqueeze(0).to(device)

          with torch.no_grad():
            output = model(tensor)
            probs = torch.softmax(output, dim=1)[0].cpu().numpy()
          
          prediction_buffer.append(probs)
          avg_probs = np.mean(prediction_buffer, axis=0)
          pred_idx = avg_probs.argmax()
          confidence = avg_probs[pred_idx]

          if confidence < 0.5:
            label = f"Uncertain ({confidence:.0%})"
            color = (128, 128, 128)
          else:
            emotion = EMOTIONS[pred_idx]
            label = f"{emotion} ({confidence:.0%})"
            color = (0, 255, 0)

          cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
          cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

          for i, (emo, prob) in enumerate(zip(EMOTIONS, probs)):
            bar_w = int(prob * 150)
            y_bar = 30 + i * 22
            cv2.rectangle(frame, (10, y_bar), (10 + bar_w, y_bar + 16), (0, 200, 100), -1)
            cv2.putText(frame, f"{emo[:3]} {prob:.0%}", (10, y_bar + 13), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

      cv2.imshow("Emotion Detector", frame)
      if cv2.waitKey(1) & 0xFF == ord('q'): break

  except Exception as e:
    print(f"Error: {e}")

  finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released")

I0000 00:00:1775159348.645114  156287 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775159348.661820  156300 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.288.01), renderer: NVIDIA GeForce GTX 1650/PCIe/SSE2
W0000 00:00:1775159348.664698  156289 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Camera released


ioctl(VIDIOC_QBUF): Bad file descriptor
